# Chapter 4: A basic ReAct agent
Classes live in `src/scratch_agents/`; this notebook uses and tests them.

## Goal (listing 4.1)
By the end of 4.6, this should run:

```python
from scratch_agents import Agent, LlmClient
from scratch_agents.tools import calculator, search_web

agent = Agent(
    model=LlmClient(model="gpt-5-mini"),
    tools=[calculator, search_web],
    instructions="You are a helpful assistant"
)

result = await agent.run("What is 1234 * 5678?")
```

## Setup
Autoreload re-reads my package files when I edit them, so no kernel restart is needed.

In [1]:
%load_ext autoreload
%autoreload 2

## 4.3 Content types (listing 4.2)
Quick check that Message builds and fills in its `type` discriminator.

In [3]:
from scratch_agents.models import Message, ToolCall, ToolResult

m = Message(role="user", content="What is 1234 * 5678?")
print(m)
print(m.type)
print(m.model_dump())

type='message' role='user' content='What is 1234 * 5678?'
message
{'type': 'message', 'role': 'user', 'content': 'What is 1234 * 5678?'}


### Break it on purpose: Pydantic validation

In [4]:
Message(role="admin", content="hi")          # a. role not in the Literal

ValidationError: 1 validation error for Message
role
  Input should be 'system', 'user' or 'assistant' [type=literal_error, input_value='admin', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/literal_error

In [5]:
Message(role="user")                          # b. content missing

ValidationError: 1 validation error for Message
content
  Field required [type=missing, input_value={'role': 'user'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing

In [6]:
ToolCall(tool_call_id="c1", name="calculator",
         arguments='{"expression": "1+1"}')  # c. arguments is a string, not a dict

ValidationError: 1 validation error for ToolCall
arguments
  Input should be a valid dictionary [type=dict_type, input_value='{"expression": "1+1"}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/dict_type

In [7]:
Message(type="tool_call", role="user", content="hi")   # d. tries to fake the discriminator

ValidationError: 1 validation error for Message
type
  Input should be 'message' [type=literal_error, input_value='tool_call', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/literal_error

## 4.3 Event and ExecutionContext (listings 4.3–4.4)

In [8]:
from scratch_agents.models import Event, ExecutionContext, Message

ctx = ExecutionContext()
e = Event(execution_id=ctx.execution_id, author="user",
          content=[Message(role="user", content="What is 1234 * 5678?")])
ctx.add_event(e)
ctx.increment_step()
print(ctx)

ExecutionContext(execution_id='20dd48c3-3cd3-464f-a6db-310f5abfd260', events=[Event(id='9e9a255e-f136-4080-af55-a2ba4f191c79', execution_id='20dd48c3-3cd3-464f-a6db-310f5abfd260', timestamp=1795505786.264036, author='user', content=[Message(type='message', role='user', content='What is 1234 * 5678?')])], current_step=1, state={}, final_result=None)
